# Brazilian E-Commerce Intelligence System
### Multi-dimensional Analysis of 100K+ Orders from Olist Marketplace

**Author:** Mehdi Hassanbeigi
**Date:** November 2024  
**Dataset:** Olist Brazilian E-commerce (2016-2018)

## Executive Summary
This comprehensive analysis examines 100K+ orders from Brazilian e-commerce platform Olist, uncovering actionable insights across customer behavior, operational efficiency, and revenue optimization.


---

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import datetime as dt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter
import re
from datetime import datetime, timedelta
import calendar
from scipy import stats
from scipy.stats import chi2_contingency
import warnings
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, r2_score
from sklearn.linear_model import LogisticRegression


try:
    import xgboost as xgb
    has_xgboost = True
except ImportError:
    has_xgboost = False
    print("XGBoost not installed. Using GradientBoostingClassifier instead.")

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

## 1. Data Acquisition and Architecture

### Understanding the Olist Ecosystem
Olist connects small businesses throughout Brazil to larger marketplaces. This dataset contains:
- **9 interconnected tables** representing the complete e-commerce workflow
- **100K+ orders** from 2016-2018
- **Multi-dimensional data:** transactional, geographical, temporal, and textual

Let's begin by loading and understanding each table's role in the ecosystem.

In [32]:
data_path = 'F:/Brazilian E-Commerce Public Dataset by Olist/'

orders = pd.read_csv(f'{data_path}olist_orders_dataset.csv')
order_items = pd.read_csv(f'{data_path}olist_order_items_dataset.csv')
customers = pd.read_csv(f'{data_path}olist_customers_dataset.csv')
products = pd.read_csv(f'{data_path}olist_products_dataset.csv')
sellers = pd.read_csv(f'{data_path}olist_sellers_dataset.csv')

reviews = pd.read_csv(f'{data_path}olist_order_reviews_dataset.csv')

payments = pd.read_csv(f'{data_path}olist_order_payments_dataset.csv')

geolocation = pd.read_csv(f'{data_path}olist_geolocation_dataset.csv')

category_translation = pd.read_csv(f'{data_path}product_category_name_translation.csv')

datasets = {
    'Orders': orders,
    'Order Items': order_items,
    'Customers': customers,
    'Products': products,
    'Sellers': sellers,
    'Reviews': reviews,
    'Payments': payments,
    'Geolocation': geolocation,
    'Category Translation': category_translation
}

for name, df in datasets.items():
    print(f"{name:20} {df.shape[0]:,} rows × {df.shape[1]} columns")

Orders               99,441 rows × 8 columns
Order Items          112,650 rows × 7 columns
Customers            99,441 rows × 5 columns
Products             32,951 rows × 9 columns
Sellers              3,095 rows × 4 columns
Reviews              99,224 rows × 7 columns
Payments             103,886 rows × 5 columns
Geolocation          1,000,163 rows × 5 columns
Category Translation 71 rows × 2 columns


## 2. Data Schema and Relationships

Understanding the relationships between tables is crucial for proper analysis. Let's visualize the data model and understand primary/foreign keys.

In [33]:
def analyze_dataframe(df, name):
    print(f" {name} Analysis")
    print('=-'*20)
    
    print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    
    print("\nColumn Types:")
    print(df.dtypes.value_counts())
    
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print("\n Missing Values:")
        missing_df = pd.DataFrame({
            'Column': missing[missing > 0].index,
            'Missing_Count': missing[missing > 0].values,
            'Percentage': (missing[missing > 0].values / len(df) * 100).round(2)
        })
        print(missing_df.to_string(index=False))
    else:
        print("\n No missing values")
    
    print(f"\n First 3 rows of {name}:")
    return df.head(3)

orders_sample = analyze_dataframe(orders, "ORDERS")
orders_sample

 ORDERS Analysis
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-

Shape: 99,441 rows × 8 columns

Column Types:
object    8
Name: count, dtype: int64

 Missing Values:
                       Column  Missing_Count  Percentage
            order_approved_at            160       0.160
 order_delivered_carrier_date           1783       1.790
order_delivered_customer_date           2965       2.980

 First 3 rows of ORDERS:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [34]:
fig = go.Figure()

entities = {
    'Orders': (0.5, 0.5),
    'Customers': (0.2, 0.7),
    'Order_Items': (0.5, 0.3),
    'Products': (0.8, 0.3),
    'Sellers': (0.8, 0.7),
    'Reviews': (0.2, 0.3),
    'Payments': (0.5, 0.7),
    'Geolocation': (0.5, 0.1),
}

for entity, (x, y) in entities.items():
    fig.add_trace(go.Scatter(
        x=[x], y=[y],
        mode='markers+text',
        text=[entity],
        textposition='middle center',
        marker=dict(size=80, color='lightblue'),
        showlegend=False,
        hovertemplate=f'<b>{entity}</b><extra></extra>'
    ))

relationships = [
    ('Orders', 'Customers', 'customer_id'),
    ('Orders', 'Order_Items', 'order_id'),
    ('Order_Items', 'Products', 'product_id'),
    ('Order_Items', 'Sellers', 'seller_id'),
    ('Orders', 'Reviews', 'order_id'),
    ('Orders', 'Payments', 'order_id'),
    ('Customers', 'Geolocation', 'zip_code'),
    ('Sellers', 'Geolocation', 'zip_code'),
]

for start, end, key in relationships:
    x0, y0 = entities[start]
    x1, y1 = entities[end]
    
    fig.add_trace(go.Scatter(
        x=[x0, x1], y=[y0, y1],
        mode='lines',
        line=dict(color='gray', width=1),
        showlegend=False,
        hoverinfo='skip'
    ))

fig.update_layout(
    title="<b>Olist E-Commerce Data Model</b>",
    title_x=0.5,
    showlegend=False,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=500,
    template='plotly_white'
)

fig.show()

print("\n Key Relationships:")
for start, end, key in relationships:
    print(f"  {start:15} → {end:15} [{key}]")


 Key Relationships:
  Orders          → Customers       [customer_id]
  Orders          → Order_Items     [order_id]
  Order_Items     → Products        [product_id]
  Order_Items     → Sellers         [seller_id]
  Orders          → Reviews         [order_id]
  Orders          → Payments        [order_id]
  Customers       → Geolocation     [zip_code]
  Sellers         → Geolocation     [zip_code]


## 3. Data Cleaning and Preprocessing

Before analysis, we need to:
1. Convert date columns to datetime
2. Handle missing values appropriately
3. Merge category translations
4. Create derived features for analysis

In [35]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col])

reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

print("\n Order Date Range:")
print(f"  First Order: {orders['order_purchase_timestamp'].min()}")
print(f"  Last Order:  {orders['order_purchase_timestamp'].max()}")
print(f"  Total Days:  {(orders['order_purchase_timestamp'].max() - orders['order_purchase_timestamp'].min()).days}")


 Order Date Range:
  First Order: 2016-09-04 21:15:19
  Last Order:  2018-10-17 17:30:18
  Total Days:  772


In [36]:
orders['delivery_time_days'] = (
    orders['order_delivered_customer_date'] - 
    orders['order_purchase_timestamp']
).dt.total_seconds() / 86400

orders['estimated_vs_actual'] = (
    orders['order_delivered_customer_date'] - 
    orders['order_estimated_delivery_date']
).dt.total_seconds() / 86400

orders['processing_time_days'] = (
    orders['order_approved_at'] - 
    orders['order_purchase_timestamp']
).dt.total_seconds() / 86400

orders['shipping_time_days'] = (
    orders['order_delivered_customer_date'] - 
    orders['order_delivered_carrier_date']
).dt.total_seconds() / 86400

orders['purchase_year'] = orders['order_purchase_timestamp'].dt.year
orders['purchase_month'] = orders['order_purchase_timestamp'].dt.month
orders['purchase_day'] = orders['order_purchase_timestamp'].dt.day
orders['purchase_weekday'] = orders['order_purchase_timestamp'].dt.dayofweek
orders['purchase_hour'] = orders['order_purchase_timestamp'].dt.hour

orders['month_name'] = orders['order_purchase_timestamp'].dt.strftime('%B')
orders['weekday_name'] = orders['order_purchase_timestamp'].dt.day_name()

new_features = ['delivery_time_days', 'estimated_vs_actual', 'processing_time_days', 'shipping_time_days']
print("\n New Features Summary:")
orders[new_features].describe().round(2)


 New Features Summary:


,delivery_time_days,estimated_vs_actual,processing_time_days,shipping_time_days
count,96476.000,96476.000,99281.000,96475.000
mean,12.560,-11.180,0.430,9.330
std,9.550,10.190,1.080,8.760
min,0.530,-146.020,0.000,-16.100
25%,6.770,-16.240,0.010,4.100
50%,10.220,-11.950,0.010,7.100
75%,15.720,-6.390,0.610,12.030
max,209.630,188.980,187.880,205.190


## 4. Master Dataset Creation

Now let's create a comprehensive master dataset by intelligently joining all tables. This will be our primary analysis dataset.

In [37]:
order_items_agg = order_items.groupby('order_id').agg({
    'product_id': 'count',
    'price': 'sum',
    'freight_value': 'sum',
    'seller_id': 'nunique'
}).rename(columns={
    'product_id': 'item_count',
    'price': 'total_price',
    'freight_value': 'total_freight',
    'seller_id': 'unique_sellers'
}).reset_index()

payments_agg = payments.groupby('order_id').agg({
    'payment_type': lambda x: x.mode()[0] if not x.empty else 'unknown',
    'payment_installments': 'max',
    'payment_value': 'sum'
}).rename(columns={
    'payment_type': 'main_payment_type',
    'payment_installments': 'max_installments',
    'payment_value': 'total_payment'
}).reset_index()

reviews_clean = reviews[['order_id', 'review_score', 'review_comment_message']].copy()
reviews_clean['has_comment'] = reviews_clean['review_comment_message'].notna().astype(int)

master = orders.merge(order_items_agg, on='order_id', how='left')
master = master.merge(payments_agg, on='order_id', how='left')
master = master.merge(reviews_clean, on='order_id', how='left')
master = master.merge(customers[['customer_id', 'customer_city', 'customer_state']], 
                     on='customer_id', how='left')

print(f"Shape: {master.shape[0]:,} rows × {master.shape[1]} columns")

master.head(3)

Shape: 99,992 rows × 31 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,estimated_vs_actual,processing_time_days,shipping_time_days,purchase_year,purchase_month,purchase_day,purchase_weekday,purchase_hour,month_name,weekday_name,item_count,total_price,total_freight,unique_sellers,main_payment_type,max_installments,total_payment,review_score,review_comment_message,has_comment,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.437,-7.107,0.007,6.063,2017,10,2,0,10,October,Monday,1.000,29.990,8.720,1.000,voucher,1.000,38.710,4.000,"Não testei o produto ainda, mas ele veio corre...",1.000,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782,-5.356,1.280,12.039,2018,7,24,1,20,July,Tuesday,1.000,118.700,22.760,1.000,boleto,1.000,141.460,4.000,Muito bom o produto.,1.000,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394,-17.245,0.012,9.178,2018,8,8,2,8,August,Wednesday,1.000,159.900,19.220,1.000,credit_card,3.000,179.120,5.000,NaN,0.000,vianopolis,GO


## 5. Exploratory Data Analysis - Business Overview

Let's start with high-level business metrics to understand the marketplace performance.

In [38]:
completed_orders = master[master['order_status'] == 'delivered']

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Total Revenue', 'Total Orders', 'Avg Order Value',
                   'Total Customers', 'Avg Review Score', 'Delivery Success Rate'),
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}],
           [{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}]]
)

total_revenue = completed_orders['total_payment'].sum()
total_orders = len(completed_orders)
avg_order_value = completed_orders['total_payment'].mean()
total_customers = master['customer_id'].nunique()
avg_review = completed_orders['review_score'].mean()
delivery_rate = (master['order_status'] == 'delivered').mean() * 100

fig.add_trace(go.Indicator(
    mode="number",
    value=total_revenue,
    number={'prefix': "R$", 'valueformat': ",.0f"},
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number",
    value=total_orders,
    number={'valueformat': ","},
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode="number",
    value=avg_order_value,
    number={'prefix': "R$", 'valueformat': ",.2f"},
), row=1, col=3)

fig.add_trace(go.Indicator(
    mode="number",
    value=total_customers,
    number={'valueformat': ","},
), row=2, col=1)

fig.add_trace(go.Indicator(
    mode="number+gauge",
    value=avg_review,
    gauge={'axis': {'range': [0, 5]}, 'bar': {'color': "darkgreen"}},
    number={'valueformat': ".2f"},
), row=2, col=2)

fig.add_trace(go.Indicator(
    mode="number+gauge",
    value=delivery_rate,
    gauge={'axis': {'range': [0, 100]}, 'bar': {'color': "darkblue"}},
    number={'suffix': "%", 'valueformat': ".1f"},
), row=2, col=3)

fig.update_layout(
    height=500,
    title_text="<b>Olist Marketplace - Key Performance Indicators</b>",
    title_x=0.5,
    showlegend=False
)

fig.show()


### Business Insights Summary
Based on the KPIs above, we can see that [you'll fill this after running the code with actual insights]

Now let's dive deeper into temporal patterns and growth trends.

## 6. Temporal Analysis - Growth & Seasonality

### Understanding Business Evolution
Let's analyze how the Olist marketplace has grown over time and identify seasonal patterns that impact business performance.

In [39]:
monthly_metrics = completed_orders.groupby(completed_orders['order_purchase_timestamp'].dt.to_period('M')).agg({
    'order_id': 'count',
    'total_payment': 'sum',
    'customer_id': 'nunique',
    'review_score': 'mean',
    'delivery_time_days': 'mean'
}).rename(columns={
    'order_id': 'orders',
    'total_payment': 'revenue',
    'customer_id': 'unique_customers',
    'review_score': 'avg_review',
    'delivery_time_days': 'avg_delivery_days'
}).reset_index()

monthly_metrics['order_purchase_timestamp'] = monthly_metrics['order_purchase_timestamp'].dt.to_timestamp()
monthly_metrics['avg_order_value'] = monthly_metrics['revenue'] / monthly_metrics['orders']

monthly_metrics['revenue_growth'] = monthly_metrics['revenue'].pct_change() * 100
monthly_metrics['order_growth'] = monthly_metrics['orders'].pct_change() * 100

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Monthly Revenue Trend', 'Monthly Order Volume', 
                   'Average Order Value Evolution', 'Customer Acquisition Trend'),
    specs=[[{'secondary_y': False}, {'secondary_y': False}],
           [{'secondary_y': False}, {'secondary_y': False}]],
    vertical_spacing=0.12,
    horizontal_spacing=0.10
)

fig.add_trace(
    go.Scatter(x=monthly_metrics['order_purchase_timestamp'], 
               y=monthly_metrics['revenue'],
               mode='lines+markers',
               name='Revenue',
               line=dict(color='#2E86AB', width=3),
               marker=dict(size=8)),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=monthly_metrics['order_purchase_timestamp'], 
           y=monthly_metrics['orders'],
           name='Orders',
           marker_color='#A23B72'),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(x=monthly_metrics['order_purchase_timestamp'], 
               y=monthly_metrics['avg_order_value'],
               mode='lines+markers',
               name='AOV',
               line=dict(color='#F18F01', width=2),
               fill='tozeroy',
               fillcolor='rgba(241, 143, 1, 0.1)'),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(x=monthly_metrics['order_purchase_timestamp'], 
               y=monthly_metrics['unique_customers'],
               mode='lines+markers',
               name='Unique Customers',
               line=dict(color='#C73E1D', width=2)),
    row=2, col=2
)

fig.update_xaxes(title_text="Month", row=2, col=1)
fig.update_xaxes(title_text="Month", row=2, col=2)
fig.update_yaxes(title_text="Revenue (R$)", row=1, col=1)
fig.update_yaxes(title_text="Orders", row=1, col=2)
fig.update_yaxes(title_text="AOV (R$)", row=2, col=1)
fig.update_yaxes(title_text="Customers", row=2, col=2)

fig.update_layout(
    height=700,
    title_text="<b>Olist Marketplace - Growth Analysis Dashboard</b>",
    title_x=0.5,
    showlegend=False,
    hovermode='x unified'
)

fig.show()



### Seasonality Patterns
Understanding weekly and hourly patterns can optimize operations and marketing efforts.

In [40]:
dow_analysis = completed_orders.groupby('weekday_name').agg({
    'order_id': 'count',
    'total_payment': 'mean',
    'review_score': 'mean'
}).round(2)

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_analysis = dow_analysis.reindex(day_order)

hour_analysis = completed_orders.groupby('purchase_hour').agg({
    'order_id': 'count',
    'total_payment': 'sum'
}).rename(columns={'order_id': 'orders', 'total_payment': 'revenue'})

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Orders by Day of Week', 'Average Order Value by Day',
                   'Orders by Hour of Day', 'Heatmap: Day vs Hour'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'scatter'}, {'type': 'heatmap'}]],
    vertical_spacing=0.15
)

fig.add_trace(
    go.Bar(x=dow_analysis.index, 
           y=dow_analysis['order_id'],
           marker_color=['#FF6B6B' if day in ['Saturday', 'Sunday'] else '#4ECDC4' 
                        for day in dow_analysis.index],
           text=dow_analysis['order_id'],
           textposition='auto'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=dow_analysis.index, 
           y=dow_analysis['total_payment'],
           marker_color='#95E77E',
           text=[f'R${val:.0f}' for val in dow_analysis['total_payment']],
           textposition='auto'),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(x=hour_analysis.index, 
              y=hour_analysis['orders'],
              mode='lines+markers',
              line=dict(color='#FFE66D', width=3),
              fill='tozeroy',
              fillcolor='rgba(255, 230, 109, 0.2)'),
    row=2, col=1
)

pivot_day_hour = completed_orders.pivot_table(
    values='order_id',
    index='purchase_hour',
    columns='weekday_name',
    aggfunc='count',
    fill_value=0
)[day_order]

fig.add_trace(
    go.Heatmap(z=pivot_day_hour.values,
              x=pivot_day_hour.columns,
              y=pivot_day_hour.index,
              colorscale='Viridis',
              showscale=True),
    row=2, col=2
)

fig.update_xaxes(title_text="Hour of Day", row=2, col=1)
fig.update_yaxes(title_text="Number of Orders", row=1, col=1)
fig.update_yaxes(title_text="Average Order Value (R$)", row=1, col=2)
fig.update_yaxes(title_text="Orders", row=2, col=1)
fig.update_xaxes(title_text="Day of Week", row=2, col=2)
fig.update_yaxes(title_text="Hour", row=2, col=2)

fig.update_layout(
    height=700,
    title_text="<b>Purchase Patterns - Temporal Behavior Analysis</b>",
    title_x=0.5,
    showlegend=False
)

fig.show()

best_day = dow_analysis['order_id'].idxmax()
best_hour = hour_analysis['orders'].idxmax()
weekend_vs_weekday = completed_orders[completed_orders['weekday_name'].isin(['Saturday', 'Sunday'])]['total_payment'].mean() / \
                     completed_orders[~completed_orders['weekday_name'].isin(['Saturday', 'Sunday'])]['total_payment'].mean()



## 7. Customer Segmentation - RFM Analysis

RFM (Recency, Frequency, Monetary) segmentation helps identify high-value customers and those at risk of churning.

In [41]:

reference_date = orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = completed_orders.groupby('customer_id').agg({
    'order_purchase_timestamp': lambda x: (reference_date - x.max()).days,
    'order_id': 'count',
    'total_payment': 'sum'
}).rename(columns={
    'order_purchase_timestamp': 'Recency',
    'order_id': 'Frequency',
    'total_payment': 'Monetary'
}).reset_index()

rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=['5','4','3','2','1'])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=['1','2','3','4','5'])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=5, labels=['1','2','3','4','5'])

rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

def segment_customers(row):
    if row['RFM_Score'] in ['555', '554', '544', '545', '454', '455', '445']:
        return 'Champions'
    elif row['RFM_Score'] in ['543', '444', '435', '355', '354', '345', '344', '335']:
        return 'Loyal Customers'
    elif row['RFM_Score'] in ['553', '551', '552', '541', '542', '533', '532', '531', '452', '451']:
        return 'Potential Loyalists'
    elif row['RFM_Score'] in ['512', '511', '422', '421', '412', '411', '311']:
        return 'New Customers'
    elif row['RFM_Score'] in ['525', '524', '523', '522', '521', '515', '514', '513', '425', '424', '413', '414', '415', '315', '314', '313']:
        return 'Promising'
    elif row['RFM_Score'] in ['535', '534', '443', '434', '343', '334', '325', '324']:
        return 'Need Attention'
    elif row['RFM_Score'] in ['331', '321', '312', '221', '213', '231', '241', '251']:
        return 'About to Sleep'
    elif row['RFM_Score'] in ['155', '154', '144', '214', '215', '115', '114', '113']:
        return 'At Risk'
    elif row['RFM_Score'] in ['255', '254', '245', '244', '253', '252', '243', '242', '235', '234', '225', '224', '153', '152', '145']:
        return 'Cannot Lose Them'
    elif row['RFM_Score'] in ['332', '322', '231', '241', '251', '233', '232', '223', '222', '132', '123', '122', '212', '211']:
        return 'Hibernating'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customers, axis=1)

segment_analysis = rfm.groupby('Segment').agg({
    'customer_id': 'count',
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean'
}).round(0).rename(columns={'customer_id': 'Count'})

segment_analysis['Percent'] = (segment_analysis['Count'] / segment_analysis['Count'].sum() * 100).round(1)
segment_analysis = segment_analysis.sort_values('Count', ascending=False)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Customer Segments Distribution', 'Monetary Value by Segment',
                   'RFM Scatter Plot', 'Segment Value Matrix'),
    specs=[[{'type': 'pie'}, {'type': 'bar'}],
           [{'type': 'scatter'}, {'type': 'bar'}]]
)

fig.add_trace(
    go.Pie(labels=segment_analysis.index,
           values=segment_analysis['Count'],
           hole=0.4,
           marker_colors=px.colors.qualitative.Set3),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=segment_analysis.index,
           y=segment_analysis['Monetary'],
           marker_color=segment_analysis['Monetary'],
           marker_colorscale='Viridis',
           text=[f'R${val:,.0f}' for val in segment_analysis['Monetary']],
           textposition='auto'),
    row=1, col=2
)

top_segments = ['Champions', 'Loyal Customers', 'At Risk', 'Cannot Lose Them', 'New Customers']
rfm_subset = rfm[rfm['Segment'].isin(top_segments)].sample(min(1000, len(rfm)))

fig.add_trace(
    go.Scatter(x=rfm_subset['Frequency'],
              y=rfm_subset['Monetary'],
              mode='markers',
              marker=dict(
                  size=5,
                  color=rfm_subset['Recency'],
                  colorscale='RdYlGn_r',
                  showscale=True,
                  colorbar=dict(title="Recency<br>(days)")
              ),
              text=rfm_subset['Segment'],
              hovertemplate='<b>%{text}</b><br>' +
                           'Frequency: %{x}<br>' +
                           'Monetary: R$%{y:,.0f}<br>' +
                           '<extra></extra>'),
    row=2, col=1
)

segment_value = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=True)
fig.add_trace(
    go.Bar(y=segment_value.index,
           x=segment_value.values,
           orientation='h',
           marker_color='lightseagreen',
           text=[f'R${val/1000:.1f}K' for val in segment_value.values],
           textposition='auto'),
    row=2, col=2
)

fig.update_xaxes(title_text="Frequency", row=2, col=1)
fig.update_yaxes(title_text="Monetary (R$)", row=2, col=1)
fig.update_xaxes(title_text="Total Revenue (R$)", row=2, col=2)

fig.update_layout(
    height=800,
    title_text="<b>Customer Segmentation - RFM Analysis</b>",
    title_x=0.5,
    showlegend=False
)

fig.show()

print("\n CUSTOMER SEGMENT SUMMARY:")
print("-"*20)
print(segment_analysis.to_string())


at_risk_segments = []
for seg in segment_analysis.index:
    if 'Risk' in seg or 'Cannot Lose' in seg:
        at_risk_segments.append(seg)

if at_risk_segments:
    at_risk_pct = segment_analysis.loc[at_risk_segments, 'Percent'].sum()
    print(f"• At Risk Groups: {at_risk_pct:.1f}% need immediate attention")

print(f"• Revenue Concentration: Top 20% customers = R${rfm.nlargest(int(len(rfm)*0.2), 'Monetary')['Monetary'].sum():,.0f}")


 CUSTOMER SEGMENT SUMMARY:
--------------------
                     Count  Recency  Frequency  Monetary  Percent
Segment                                                          
Lost                 20881  373.000      1.000   104.000   21.600
Promising            12467  157.000      1.000   215.000   12.900
Cannot Lose Them     11639  399.000      1.000   223.000   12.100
Hibernating           8581  395.000      1.000    76.000    8.900
Potential Loyalists   7379  112.000      1.000    65.000    7.600
Need Attention        6302  207.000      1.000   206.000    6.500
Loyal Customers       6292  227.000      1.000   279.000    6.500
About to Sleep        6106  329.000      1.000    52.000    6.300
At Risk               5938  485.000      1.000   257.000    6.200
Champions             5470  136.000      1.000   317.000    5.700
New Customers         5423  174.000      1.000    52.000    5.600
• At Risk Groups: 18.3% need immediate attention
• Revenue Concentration: Top 20% customers =

### Strategic Recommendations by Segment

Based on the RFM analysis, here are targeted strategies for each segment:

1. **Champions**: VIP treatment, early access to new products
2. **At Risk**: Win-back campaigns with special offers
3. **New Customers**: Onboarding sequence to increase engagement
4. **Cannot Lose Them**: Personal attention, satisfaction surveys

## 8. Geographic Analysis - Brazilian Market Intelligence

Understanding geographic distribution helps optimize logistics, identify growth opportunities, and improve delivery performance.

In [42]:
state_analysis = completed_orders.groupby('customer_state').agg({
    'order_id': 'count',
    'total_payment': 'sum',
    'delivery_time_days': 'mean',
    'review_score': 'mean',
    'customer_id': 'nunique'
}).rename(columns={
    'order_id': 'orders',
    'total_payment': 'revenue',
    'delivery_time_days': 'avg_delivery_days',
    'review_score': 'avg_review',
    'customer_id': 'customers'
}).round(2)

state_analysis['avg_order_value'] = (state_analysis['revenue'] / state_analysis['orders']).round(2)
state_analysis['orders_per_customer'] = (state_analysis['orders'] / state_analysis['customers']).round(2)
state_analysis = state_analysis.sort_values('revenue', ascending=False)

top_states = state_analysis.head(10)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Revenue by State (Top 10)', 'Delivery Performance by State',
                   'Customer Distribution', 'Revenue vs Delivery Time'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'pie'}, {'type': 'scatter'}]]
)

fig.add_trace(
    go.Bar(x=top_states.index,
           y=top_states['revenue'],
           marker_color='steelblue',
           text=[f'R${val/1e6:.1f}M' for val in top_states['revenue']],
           textposition='auto'),
    row=1, col=1
)

delivery_colors = ['green' if days < 10 else 'orange' if days < 15 else 'red' 
                   for days in top_states['avg_delivery_days']]
fig.add_trace(
    go.Bar(x=top_states.index,
           y=top_states['avg_delivery_days'],
           marker_color=delivery_colors,
           text=[f'{val:.1f} days' for val in top_states['avg_delivery_days']],
           textposition='auto'),
    row=1, col=2
)

fig.add_trace(
    go.Pie(labels=top_states.index,
           values=top_states['customers'],
           hole=0.3),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(x=state_analysis['avg_delivery_days'],
              y=state_analysis['revenue'],
              mode='markers+text',
              text=state_analysis.index,
              textposition='top center',
              marker=dict(
                  size=state_analysis['orders']/100,
                  color=state_analysis['avg_review'],
                  colorscale='RdYlGn',
                  showscale=True,
                  colorbar=dict(title="Avg<br>Review"),
                  sizemode='diameter'
              ),
              hovertemplate='<b>%{text}</b><br>' +
                           'Revenue: R$%{y:,.0f}<br>' +
                           'Avg Delivery: %{x:.1f} days<br>' +
                           '<extra></extra>'),
    row=2, col=2
)

fig.update_xaxes(title_text="State", row=1, col=1)
fig.update_xaxes(title_text="State", row=1, col=2)
fig.update_xaxes(title_text="Average Delivery Days", row=2, col=2)
fig.update_yaxes(title_text="Revenue (R$)", row=1, col=1)
fig.update_yaxes(title_text="Days", row=1, col=2)
fig.update_yaxes(title_text="Revenue (R$)", row=2, col=2)

fig.update_layout(
    height=700,
    title_text="<b>Geographic Performance Analysis</b>",
    title_x=0.5,
    showlegend=False
)

fig.show()

sp_dominance = (state_analysis.loc['SP', 'revenue'] / state_analysis['revenue'].sum()) * 100
avg_delivery_sp = state_analysis.loc['SP', 'avg_delivery_days']
worst_delivery_state = state_analysis.nlargest(10, 'revenue')['avg_delivery_days'].idxmax()



### City-Level Deep Dive
Let's analyze the top performing cities and identify urban vs rural patterns.

In [43]:
city_analysis = completed_orders.groupby(['customer_state', 'customer_city']).agg({
    'order_id': 'count',
    'total_payment': 'sum',
    'delivery_time_days': 'mean',
    'review_score': 'mean'
}).rename(columns={
    'order_id': 'orders',
    'total_payment': 'revenue',
    'delivery_time_days': 'avg_delivery',
    'review_score': 'avg_review'
}).round(2)

city_analysis = city_analysis.reset_index()
city_analysis['city_state'] = city_analysis['customer_city'] + ', ' + city_analysis['customer_state']
top_cities = city_analysis.nlargest(20, 'revenue')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Top 20 Cities by Revenue', 'Delivery Time Distribution'),
    specs=[[{'type': 'bar'}, {'type': 'box'}]]
)

fig.add_trace(
    go.Bar(y=top_cities['city_state'][::-1],
           x=top_cities['revenue'][::-1],
           orientation='h',
           marker_color='teal',
           text=[f'R${val/1000:.0f}K' for val in top_cities['revenue'][::-1]],
           textposition='auto'),
    row=1, col=1
)

top_5_states = state_analysis.head(5).index
for state in top_5_states:
    state_delivery = completed_orders[completed_orders['customer_state'] == state]['delivery_time_days']
    fig.add_trace(
        go.Box(y=state_delivery,
               name=state,
               boxpoints='outliers'),
        row=1, col=2
    )

fig.update_xaxes(title_text="Revenue (R$)", row=1, col=1)
fig.update_yaxes(title_text="City", row=1, col=1)
fig.update_yaxes(title_text="Delivery Days", row=1, col=2)

fig.update_layout(
    height=500,
    title_text="<b>City-Level Market Analysis</b>",
    title_x=0.5,
    showlegend=True
)

fig.show()

capitals = ['sao paulo', 'rio de janeiro', 'brasilia', 'belo horizonte', 'curitiba', 
            'porto alegre', 'salvador', 'fortaleza', 'recife', 'belem']

city_analysis['is_capital'] = city_analysis['customer_city'].str.lower().isin(capitals)
capital_comparison = city_analysis.groupby('is_capital').agg({
    'orders': 'sum',
    'revenue': 'sum',
    'avg_delivery': 'mean'
}).round(2)



## 9. Delivery Performance Analysis

Delivery is crucial for customer satisfaction in e-commerce. Let's analyze delivery performance patterns and identify improvement opportunities.

In [44]:
delivery_analysis = completed_orders.copy()
delivery_analysis = delivery_analysis[delivery_analysis['delivery_time_days'].notna()]

def categorize_delivery(days):
    if pd.isna(days):
        return 'Unknown'
    elif days <= 5:
        return 'Excellent (≤5 days)'
    elif days <= 10:
        return 'Good (6-10 days)'
    elif days <= 15:
        return 'Average (11-15 days)'
    elif days <= 20:
        return 'Poor (16-20 days)'
    else:
        return 'Very Poor (>20 days)'

delivery_analysis['delivery_category'] = delivery_analysis['delivery_time_days'].apply(categorize_delivery)

delivery_analysis['on_time'] = delivery_analysis['estimated_vs_actual'] <= 0
on_time_rate = delivery_analysis['on_time'].mean() * 100

delivery_cat_analysis = delivery_analysis.groupby('delivery_category').agg({
    'order_id': 'count',
    'review_score': 'mean',
    'total_payment': 'mean'
}).rename(columns={
    'order_id': 'orders',
    'review_score': 'avg_review',
    'total_payment': 'avg_order_value'
}).round(2)

cat_order = ['Excellent (≤5 days)', 'Good (6-10 days)', 'Average (11-15 days)', 
             'Poor (16-20 days)', 'Very Poor (>20 days)']
delivery_cat_analysis = delivery_cat_analysis.reindex([c for c in cat_order if c in delivery_cat_analysis.index])

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Delivery Time Distribution', 'Delivery Performance Categories', 
                   'Review Score by Delivery Time', 'On-Time Delivery Trend',
                   'Delivery Time vs Order Value', 'State Delivery Heatmap'),
    specs=[[{'type': 'histogram'}, {'type': 'pie'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'scatter'}, {'type': 'bar'}]]
)

fig.add_trace(
    go.Histogram(x=delivery_analysis['delivery_time_days'],
                nbinsx=50,
                marker_color='lightblue',
                name='Delivery Days'),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=delivery_cat_analysis.index,
           values=delivery_cat_analysis['orders'],
           hole=0.3,
           marker_colors=['green', 'lightgreen', 'yellow', 'orange', 'red']),
    row=1, col=2
)

sample_delivery = delivery_analysis.sample(min(5000, len(delivery_analysis)))
fig.add_trace(
    go.Scatter(x=sample_delivery['delivery_time_days'],
              y=sample_delivery['review_score'],
              mode='markers',
              marker=dict(size=3, color=sample_delivery['review_score'], 
                         colorscale='RdYlGn', showscale=False),
              opacity=0.5),
    row=1, col=3
)

monthly_ontime = delivery_analysis.groupby(delivery_analysis['order_purchase_timestamp'].dt.to_period('M'))['on_time'].mean() * 100
monthly_ontime.index = monthly_ontime.index.to_timestamp()

fig.add_trace(
    go.Scatter(x=monthly_ontime.index,
              y=monthly_ontime.values,
              mode='lines+markers',
              line=dict(color='purple', width=2),
              marker=dict(size=8)),
    row=2, col=1
)

delivery_value_bins = pd.qcut(delivery_analysis['delivery_time_days'], q=10)
delivery_value_analysis = delivery_analysis.groupby(delivery_value_bins)['total_payment'].mean()

fig.add_trace(
    go.Scatter(x=[f'{interval.left:.0f}-{interval.right:.0f}' for interval in delivery_value_analysis.index],
              y=delivery_value_analysis.values,
              mode='lines+markers',
              line=dict(color='orange', width=3),
              marker=dict(size=10)),
    row=2, col=2
)

state_delivery_perf = delivery_analysis.groupby('customer_state')['delivery_time_days'].mean().sort_values().head(15)
colors = ['green' if x < 10 else 'orange' if x < 15 else 'red' for x in state_delivery_perf.values]

fig.add_trace(
    go.Bar(x=state_delivery_perf.index,
           y=state_delivery_perf.values,
           marker_color=colors,
           text=[f'{val:.1f}' for val in state_delivery_perf.values],
           textposition='auto'),
    row=2, col=3
)

fig.update_xaxes(title_text="Delivery Days", row=1, col=1)
fig.update_xaxes(title_text="Delivery Days", row=1, col=3)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Review Score", row=1, col=3)
fig.update_xaxes(title_text="Month", row=2, col=1)
fig.update_yaxes(title_text="On-Time %", row=2, col=1)
fig.update_xaxes(title_text="Delivery Days Range", row=2, col=2)
fig.update_yaxes(title_text="Avg Order Value (R$)", row=2, col=2)
fig.update_xaxes(title_text="State", row=2, col=3)
fig.update_yaxes(title_text="Avg Delivery Days", row=2, col=3)

fig.update_layout(
    height=700,
    title_text="<b>Delivery Performance Deep Dive</b>",
    title_x=0.5,
    showlegend=False
)

fig.show()

print(delivery_cat_analysis.to_string())
print(f"On-time delivery rate: {on_time_rate:.1f}%")
print(f"Average delivery time: {delivery_analysis['delivery_time_days'].mean():.1f} days")
print(f"Median delivery time: {delivery_analysis['delivery_time_days'].median():.1f} days")
print(f"Delivery impact on reviews: {delivery_analysis[delivery_analysis['delivery_time_days'] <= 5]['review_score'].mean():.2f} (≤5 days) vs {delivery_analysis[delivery_analysis['delivery_time_days'] > 15]['review_score'].mean():.2f} (>15 days)")

delivery_review_corr = delivery_analysis[['delivery_time_days', 'review_score']].corr().iloc[0, 1]
print(f"Delivery-Review correlation: {delivery_review_corr:.3f}")

                      orders  avg_review  avg_order_value
delivery_category                                        
Excellent (≤5 days)    13483       4.450          129.810
Good (6-10 days)       33206       4.360          151.300
Average (11-15 days)   23753       4.270          166.040
Poor (16-20 days)      12238       4.130          173.900
Very Poor (>20 days)   14319       3.230          184.700
On-time delivery rate: 91.9%
Average delivery time: 12.6 days
Median delivery time: 10.2 days
Delivery impact on reviews: 4.45 (≤5 days) vs 3.65 (>15 days)
Delivery-Review correlation: -0.334


## 10. Product & Category Analysis

Understanding product performance and category trends helps optimize inventory, pricing, and marketing strategies.

In [45]:
products_translated = products.merge(category_translation, on='product_category_name', how='left')
products_translated['category_english'] = products_translated['product_category_name_english'].fillna(products_translated['product_category_name'])

product_analysis = order_items.merge(products_translated, on='product_id', how='left')
product_analysis = product_analysis.merge(orders[['order_id', 'order_status', 'order_purchase_timestamp']], on='order_id')
product_analysis = product_analysis.merge(reviews[['order_id', 'review_score']], on='order_id', how='left')

product_analysis = product_analysis[product_analysis['order_status'] == 'delivered']

print(f"Product analysis dataset ready: {len(product_analysis):,} order items")
print(f"Unique products: {product_analysis['product_id'].nunique():,}")
print(f"Product categories: {product_analysis['category_english'].nunique()}")

Product analysis dataset ready: 110,840 order items
Unique products: 32,216
Product categories: 73


In [46]:
category_performance = product_analysis.groupby('category_english').agg({
    'order_id': 'count',
    'price': 'sum',
    'product_id': 'nunique',
    'review_score': 'mean',
    'freight_value': 'mean'
}).rename(columns={
    'order_id': 'orders',
    'price': 'revenue',
    'product_id': 'unique_products',
    'review_score': 'avg_review',
    'freight_value': 'avg_freight'
}).round(2)

category_performance['avg_price'] = (category_performance['revenue'] / category_performance['orders']).round(2)
category_performance = category_performance.sort_values('revenue', ascending=False)

top_categories = category_performance.head(15)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Top 15 Categories by Revenue', 'Category Order Distribution',
                   'Avg Price by Category', 'Review Score vs Revenue'),
    specs=[[{'type': 'bar'}, {'type': 'pie'}],
           [{'type': 'bar'}, {'type': 'scatter'}]]
)

fig.add_trace(
    go.Bar(x=top_categories['revenue'],
           y=top_categories.index[::-1],
           orientation='h',
           marker_color='steelblue',
           text=[f'R${val/1000:.0f}K' for val in top_categories['revenue'][::-1]],
           textposition='auto'),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=top_categories.index[:10],
           values=top_categories['orders'][:10],
           hole=0.3),
    row=1, col=2
)

fig.add_trace(
    go.Bar(x=top_categories['avg_price'],
           y=top_categories.index[::-1],
           orientation='h',
           marker_color='coral',
           text=[f'R${val:.0f}' for val in top_categories['avg_price'][::-1]],
           textposition='auto'),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(x=category_performance['revenue'],
              y=category_performance['avg_review'],
              mode='markers',
              text=category_performance.index,
              marker=dict(
                  size=category_performance['orders']/100,
                  color=category_performance['avg_price'],
                  colorscale='Viridis',
                  showscale=True,
                  colorbar=dict(title="Avg<br>Price")
              ),
              hovertemplate='<b>%{text}</b><br>' +
                           'Revenue: R$%{x:,.0f}<br>' +
                           'Avg Review: %{y:.2f}<br>' +
                           '<extra></extra>'),
    row=2, col=2
)

fig.update_xaxes(title_text="Revenue (R$)", row=1, col=1)
fig.update_xaxes(title_text="Average Price (R$)", row=2, col=1)
fig.update_xaxes(title_text="Revenue (R$)", row=2, col=2, type='log')
fig.update_yaxes(title_text="Category", row=1, col=1)
fig.update_yaxes(title_text="Category", row=2, col=1)
fig.update_yaxes(title_text="Average Review Score", row=2, col=2)

fig.update_layout(
    height=800,
    title_text="<b>Product Category Performance Analysis</b>",
    title_x=0.5,
    showlegend=False
)

fig.show()

print(f"Top category: {top_categories.index[0]} - R${top_categories.iloc[0]['revenue']/1e6:.2f}M revenue")
print(f"Most reviewed category: {category_performance.nlargest(1, 'avg_review').index[0]} ({category_performance['avg_review'].max():.2f}/5)")
print(f"Highest AOV category: {category_performance.nlargest(1, 'avg_price').index[0]} (R${category_performance['avg_price'].max():.2f})")
print(f"Most diverse category: {category_performance.nlargest(1, 'unique_products').index[0]} ({category_performance['unique_products'].max():.0f} products)")

Top category: health_beauty - R$1.24M revenue
Most reviewed category: fashion_childrens_clothes (5.00/5)
Highest AOV category: computers (R$1098.92)
Most diverse category: bed_bath_table (2991 products)


In [47]:
product_performance = product_analysis.groupby('product_id').agg({
    'order_id': 'count',
    'price': ['sum', 'mean'],
    'review_score': 'mean',
    'category_english': 'first'
}).round(2)

product_performance.columns = ['orders', 'total_revenue', 'avg_price', 'avg_review', 'category']
product_performance = product_performance.sort_values('total_revenue', ascending=False)

top_products = product_performance.head(20)

top_20_revenue_share = (top_products['total_revenue'].sum() / product_performance['total_revenue'].sum()) * 100
top_100_revenue_share = (product_performance.head(100)['total_revenue'].sum() / product_performance['total_revenue'].sum()) * 100

cumulative_revenue = product_performance['total_revenue'].cumsum() / product_performance['total_revenue'].sum() * 100
products_for_80_percent = (cumulative_revenue <= 80).sum()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Product Revenue Concentration (Pareto)', 'Top 20 Products Performance'),
    specs=[[{'type': 'scatter'}, {'type': 'bar'}]]
)

sample_size = min(1000, len(product_performance))
x_vals = np.arange(1, sample_size + 1)
y_vals = product_performance['total_revenue'].iloc[:sample_size].cumsum() / product_performance['total_revenue'].sum() * 100

fig.add_trace(
    go.Scatter(x=x_vals, 
              y=y_vals,
              mode='lines',
              line=dict(color='darkblue', width=3),
              fill='tozeroy',
              fillcolor='rgba(30, 60, 180, 0.1)'),
    row=1, col=1
)

fig.add_hline(y=80, line_dash="dash", line_color="red", 
             annotation_text="80% Revenue", row=1, col=1)

fig.add_trace(
    go.Bar(y=np.arange(len(top_products)),
           x=top_products['total_revenue'],
           orientation='h',
           text=[f'R${val/1000:.0f}K' for val in top_products['total_revenue']],
           textposition='auto',
           marker_color='teal'),
    row=1, col=2
)

fig.update_xaxes(title_text="Number of Products", row=1, col=1)
fig.update_yaxes(title_text="Cumulative Revenue %", row=1, col=1)
fig.update_xaxes(title_text="Revenue (R$)", row=1, col=2)
fig.update_yaxes(title_text="Product Rank", row=1, col=2)

fig.update_layout(
    height=400,
    title_text="<b>Product Concentration Analysis</b>",
    title_x=0.5,
    showlegend=False
)

fig.show()

print(f"Top 20 products: {top_20_revenue_share:.1f}% of revenue")
print(f"Top 100 products: {top_100_revenue_share:.1f}% of revenue")
print(f"Products for 80% revenue: {products_for_80_percent:,} ({products_for_80_percent/len(product_performance)*100:.1f}%)")
print(f"Total active products: {len(product_performance):,}")

Top 20 products: 5.4% of revenue
Top 100 products: 13.6% of revenue
Products for 80% revenue: 8,356 (25.9%)
Total active products: 32,216


## 11. Seasonal & Trend Analysis by Category

Understanding seasonal patterns helps optimize inventory and marketing campaigns.

In [48]:
product_analysis['year_month'] = pd.to_datetime(product_analysis['order_purchase_timestamp']).dt.to_period('M')

top_5_categories = category_performance.head(5).index

monthly_category_trends = product_analysis[product_analysis['category_english'].isin(top_5_categories)].groupby(
    ['year_month', 'category_english']
)['price'].sum().reset_index()

monthly_category_trends['year_month'] = monthly_category_trends['year_month'].dt.to_timestamp()

fig = go.Figure()

for category in top_5_categories:
    category_data = monthly_category_trends[monthly_category_trends['category_english'] == category]
    fig.add_trace(go.Scatter(
        x=category_data['year_month'],
        y=category_data['price'],
        mode='lines+markers',
        name=category,
        line=dict(width=2),
        marker=dict(size=6)
    ))

fig.update_layout(
    title="<b>Top 5 Categories - Revenue Trends</b>",
    xaxis_title="Month",
    yaxis_title="Revenue (R$)",
    height=500,
    hovermode='x unified',
    legend=dict(orientation="h", y=-0.15)
)

fig.show()

product_analysis['month'] = pd.to_datetime(product_analysis['order_purchase_timestamp']).dt.month
seasonal_categories = product_analysis.groupby(['month', 'category_english'])['price'].sum().reset_index()

category_seasonality = {}
for category in top_5_categories:
    cat_data = seasonal_categories[seasonal_categories['category_english'] == category]
    if len(cat_data) > 0:
        seasonality_score = cat_data['price'].std() / cat_data['price'].mean()
        category_seasonality[category] = seasonality_score

sorted_seasonality = sorted(category_seasonality.items(), key=lambda x: x[1], reverse=True)
for category, score in sorted_seasonality[:5]:
    print(f"{category}: Seasonality score = {score:.3f}")

health_beauty: Seasonality score = 0.385
computers_accessories: Seasonality score = 0.365
watches_gifts: Seasonality score = 0.323
bed_bath_table: Seasonality score = 0.292
sports_leisure: Seasonality score = 0.243


## 12. Review Text Analysis & Sentiment Mining

Let's analyze customer reviews to understand satisfaction drivers and pain points.

In [49]:
reviews_with_text = reviews[reviews['review_comment_message'].notna()].copy()

reviews_with_text['comment_length'] = reviews_with_text['review_comment_message'].str.len()
reviews_with_text['word_count'] = reviews_with_text['review_comment_message'].str.split().str.len()

reviews_full = reviews_with_text.merge(
    orders[['order_id', 'order_status', 'delivery_time_days']], 
    on='order_id', 
    how='left'
)

review_text_analysis = reviews_with_text.groupby('review_score').agg({
    'review_id': 'count',
    'comment_length': 'mean',
    'word_count': 'mean'
}).rename(columns={
    'review_id': 'count',
    'comment_length': 'avg_char_length',
    'word_count': 'avg_word_count'
}).round(1)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Review Score Distribution', 'Comment Length by Score',
                   'Word Count by Score', 'Reviews with Comments %'),
    specs=[[{'type': 'bar'}, {'type': 'box'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

score_dist = reviews['review_score'].value_counts().sort_index()
colors = ['red' if s <= 2 else 'orange' if s == 3 else 'green' for s in score_dist.index]

fig.add_trace(
    go.Bar(x=score_dist.index,
           y=score_dist.values,
           marker_color=colors,
           text=score_dist.values,
           textposition='auto'),
    row=1, col=1
)

for score in [1, 2, 3, 4, 5]:
    score_data = reviews_with_text[reviews_with_text['review_score'] == score]['comment_length']
    fig.add_trace(
        go.Box(y=score_data,
               name=str(score),
               showlegend=False),
        row=1, col=2
    )

fig.add_trace(
    go.Bar(x=review_text_analysis.index,
           y=review_text_analysis['avg_word_count'],
           marker_color=['red' if s <= 2 else 'orange' if s == 3 else 'green' 
                        for s in review_text_analysis.index],
           text=[f'{val:.0f}' for val in review_text_analysis['avg_word_count']],
           textposition='auto'),
    row=2, col=1
)

comment_pct = reviews.groupby('review_score').apply(
    lambda x: (x['review_comment_message'].notna().sum() / len(x)) * 100
)

fig.add_trace(
    go.Bar(x=comment_pct.index,
           y=comment_pct.values,
           marker_color='lightblue',
           text=[f'{val:.1f}%' for val in comment_pct.values],
           textposition='auto'),
    row=2, col=2
)

fig.update_xaxes(title_text="Review Score", row=1, col=1)
fig.update_xaxes(title_text="Review Score", row=2, col=1)
fig.update_xaxes(title_text="Review Score", row=2, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Character Length", row=1, col=2)
fig.update_yaxes(title_text="Average Words", row=2, col=1)
fig.update_yaxes(title_text="% with Comments", row=2, col=2)

fig.update_layout(
    height=700,
    title_text="<b>Review Text Analysis</b>",
    title_x=0.5,
    showlegend=False
)

fig.show()

print(review_text_analysis.to_string())
print(f"Negative reviews (1-2 stars): {review_text_analysis.loc[[1,2], 'avg_word_count'].mean():.0f} words avg")
print(f"Positive reviews (4-5 stars): {review_text_analysis.loc[[4,5], 'avg_word_count'].mean():.0f} words avg")
print(f"Most talkative: {review_text_analysis['avg_word_count'].idxmax()}-star reviews")

              count  avg_char_length  avg_word_count
review_score                                        
1              8745           99.900          17.500
2              2145           96.800          17.100
3              3557           83.300          14.600
4              5976           61.700          10.600
5             20554           51.800           8.400
Negative reviews (1-2 stars): 17 words avg
Positive reviews (4-5 stars): 10 words avg
Most talkative: 1-star reviews


In [50]:


def extract_common_words(text_series, min_length=4):
    all_text = ' '.join(text_series.dropna().str.lower())
    
    all_text = re.sub(r'[^a-z\s]', ' ', all_text)
    
    words = [word for word in all_text.split() if len(word) >= min_length]
    
    return Counter(words).most_common(20)

negative_reviews = reviews_with_text[reviews_with_text['review_score'] <= 2]
positive_reviews = reviews_with_text[reviews_with_text['review_score'] >= 4]

print("TOP WORDS IN NEGATIVE REVIEWS (1-2 stars):")
print("-" * 40)
negative_words = extract_common_words(negative_reviews['review_comment_message'])
for word, count in negative_words[:10]:
    print(f"'{word}': {count} times")

print("\nTOP WORDS IN POSITIVE REVIEWS (4-5 stars):")
print("-" * 40)
positive_words = extract_common_words(positive_reviews['review_comment_message'])
for word, count in positive_words[:10]:
    print(f"'{word}': {count} times")

category_reviews = product_analysis.merge(
    reviews_with_text[['order_id', 'review_comment_message']], 
    on='order_id',
    how='inner'
)

category_review_summary = category_reviews.groupby('category_english').agg({
    'review_score': ['mean', 'count'],
    'review_comment_message': lambda x: x.notna().sum()
}).round(2)

category_review_summary.columns = ['avg_score', 'review_count', 'comments_count']
category_review_summary['comment_rate'] = (category_review_summary['comments_count'] / category_review_summary['review_count'] * 100).round(1)
category_review_summary = category_review_summary[category_review_summary['review_count'] > 100].sort_values('avg_score', ascending=False)

print("\nBEST REVIEWED CATEGORIES (>100 reviews):")
print("-" * 50)
print(category_review_summary.head(5).to_string())

print("\nWORST REVIEWED CATEGORIES (>100 reviews):")
print("-" * 50)
print(category_review_summary.tail(5).to_string())

TOP WORDS IN NEGATIVE REVIEWS (1-2 stars):
----------------------------------------
'produto': 6189 times
'recebi': 3273 times
'comprei': 1882 times
'para': 1642 times
'veio': 1552 times
'entrega': 1412 times
'ainda': 1401 times
'entregue': 1329 times
'chegou': 1180 times
'estou': 961 times

TOP WORDS IN POSITIVE REVIEWS (4-5 stars):
----------------------------------------
'produto': 10646 times
'prazo': 7077 times
'muito': 6548 times
'antes': 5281 times
'entrega': 4633 times
'chegou': 4016 times
'recomendo': 3873 times
'entregue': 2165 times
'qualidade': 2162 times
'tudo': 2094 times

BEST REVIEWED CATEGORIES (>100 reviews):
--------------------------------------------------
                          avg_score  review_count  comments_count  comment_rate
category_english                                                               
books_general_interest        4.230           167             167       100.000
fashion_shoes                 4.120           132             132       10

## 13. Final Business Insights Summary

Based on our comprehensive analysis, here are the key findings and recommendations.

In [51]:
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=(
        'Revenue Growth Trajectory', 'Customer Segment Value', 'Geographic Revenue Distribution',
        'Category Performance Matrix', 'Delivery Impact on Satisfaction', 'Review Score Trend',
        'Order Volume Heatmap', 'Payment Method Distribution', 'Seller Concentration'
    ),
    specs=[[{'type': 'scatter'}, {'type': 'pie'}, {'type': 'bar'}],
           [{'type': 'scatter'}, {'type': 'scatter'}, {'type': 'scatter'}],
           [{'type': 'heatmap'}, {'type': 'pie'}, {'type': 'bar'}]]
)

fig.add_trace(
    go.Scatter(x=monthly_metrics['order_purchase_timestamp'],
              y=monthly_metrics['revenue'],
              mode='lines+markers',
              line=dict(color='blue', width=2),
              fill='tozeroy',
              fillcolor='rgba(0,100,255,0.1)'),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=segment_analysis.index[:5],
           values=segment_analysis['Count'][:5],
           hole=0.4),
    row=1, col=2
)

fig.add_trace(
    go.Bar(x=top_states.index[:5],
           y=top_states['revenue'][:5],
           marker_color='green'),
    row=1, col=3
)

fig.add_trace(
    go.Scatter(x=top_categories['orders'][:10],
              y=top_categories['avg_price'][:10],
              mode='markers+text',
              text=top_categories.index[:10],
              textposition='top center',
              marker=dict(size=top_categories['revenue'][:10]/50000, color='orange')),
    row=2, col=1
)

delivery_satisfaction = delivery_analysis.groupby(
    pd.cut(delivery_analysis['delivery_time_days'], bins=5)
)['review_score'].mean()

fig.add_trace(
    go.Scatter(x=[str(i) for i in delivery_satisfaction.index],
              y=delivery_satisfaction.values,
              mode='lines+markers',
              line=dict(color='red', width=3),
              marker=dict(size=10)),
    row=2, col=2
)

review_trend = completed_orders.groupby(
    completed_orders['order_purchase_timestamp'].dt.to_period('M')
)['review_score'].mean()
review_trend.index = review_trend.index.to_timestamp()

fig.add_trace(
    go.Scatter(x=review_trend.index,
              y=review_trend.values,
              mode='lines',
              line=dict(color='green', width=2)),
    row=2, col=3
)

heatmap_data = pivot_day_hour.iloc[:12, :5].values

fig.add_trace(
    go.Heatmap(z=heatmap_data,
              colorscale='Blues'),
    row=3, col=1
)

payment_dist = payments['payment_type'].value_counts()
fig.add_trace(
    go.Pie(labels=payment_dist.index[:5],
           values=payment_dist.values[:5]),
    row=3, col=2
)

seller_orders = order_items.groupby('seller_id')['order_id'].count().sort_values(ascending=False)
top_sellers_share = seller_orders.head(20).sum() / seller_orders.sum() * 100

fig.add_trace(
    go.Bar(x=['Top 20', 'Others'],
           y=[top_sellers_share, 100-top_sellers_share],
           marker_color=['darkblue', 'lightblue']),
    row=3, col=3
)

fig.update_layout(
    height=900,
    title_text="<b>E-Commerce Analytics - Executive Dashboard</b>",
    title_x=0.5,
    showlegend=False
)

for i in range(1, 4):
    for j in range(1, 4):
        fig.update_xaxes(showticklabels=False, row=i, col=j)
        fig.update_yaxes(showticklabels=False, row=i, col=j)

fig.show()


## 14. Predictive Analytics

Building machine learning models to predict customer behavior and business outcomes.

### Models Implemented:
1. **Customer Churn Prediction** - Identify at-risk customers
2. **Customer Lifetime Value (CLV)** - Predict future customer value
3. **Demand Forecasting** - Predict future sales trends

In [52]:

ml_data = rfm.copy()

customer_features = completed_orders.groupby('customer_id').agg({
    'total_payment': ['mean', 'std', 'min', 'max'],
    'item_count': ['mean', 'sum'],
    'review_score': ['mean', 'std'],
    'delivery_time_days': ['mean', 'std'],
    'estimated_vs_actual': 'mean',
    'unique_sellers': 'mean',
    'total_freight': 'mean'
}).round(2)

customer_features.columns = ['_'.join(col).strip() for col in customer_features.columns]
customer_features = customer_features.reset_index()

ml_data = ml_data.merge(customer_features, on='customer_id', how='left')

numeric_columns = ml_data.select_dtypes(include=[np.number]).columns
for col in numeric_columns:
    if ml_data[col].isnull().any():
        ml_data[col] = ml_data[col].fillna(ml_data[col].median())

ml_data = ml_data.drop(['R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'Segment'], axis=1, errors='ignore')

print(f"ML dataset ready: {ml_data.shape}")
print(f"Features: {ml_data.shape[1] - 1}")
print(f"Samples: {ml_data.shape[0]:,}")

ML dataset ready: (96478, 17)
Features: 16
Samples: 96,478


In [53]:
churn_threshold = 60
ml_data['is_churned'] = (ml_data['Recency'] > churn_threshold).astype(int)

print(f"Churn Rate: {ml_data['is_churned'].mean():.1%}")
print(f"Churned customers: {ml_data['is_churned'].sum():,}")
print(f"Active customers: {(1-ml_data['is_churned']).sum():,}")

feature_cols = ['Frequency', 'Monetary', 'Recency', 
                'total_payment_mean', 'total_payment_std',
                'item_count_mean', 'review_score_mean',
                'delivery_time_days_mean', 'unique_sellers_mean']

feature_cols = [col for col in feature_cols if col in ml_data.columns]

X_churn = ml_data[feature_cols]
y_churn = ml_data['is_churned']

X_train, X_test, y_train, y_test = train_test_split(
    X_churn, y_churn, test_size=0.2, random_state=42, stratify=y_churn
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
}

if has_xgboost:
    models['XGBoost'] = xgb.XGBClassifier(random_state=42, n_estimators=100, max_depth=5, learning_rate=0.1)
else:
    models['Gradient Boosting'] = GradientBoostingClassifier(random_state=42, n_estimators=100, max_depth=5, learning_rate=0.1)

results = {}

for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    }

results_df = pd.DataFrame(results).T
print("\nModel Performance Comparison:")
print("-"*20)
print(results_df.round(3))

best_model_name = results_df['F1-Score'].idxmax()
best_model = models[best_model_name]

if best_model_name == 'Logistic Regression':
    best_model.fit(X_train_scaled, y_train)
    y_pred_best = best_model.predict(X_test_scaled)
else:
    best_model.fit(X_train, y_train)
    y_pred_best = best_model.predict(X_test)

if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
else:
    feature_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': np.abs(best_model.coef_[0])
    }).sort_values('importance', ascending=False)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Model Performance Comparison', 'Feature Importance', 'Confusion Matrix')
)

fig.add_trace(
    go.Bar(x=results_df.index,
           y=results_df['F1-Score'],
           marker_color=['lightblue', 'lightgreen', 'coral'][:len(results_df)],
           text=[f'{v:.3f}' for v in results_df['F1-Score']],
           textposition='auto'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=feature_importance['importance'][:8],
           y=feature_importance['feature'][:8],
           orientation='h',
           marker_color='teal',
           text=[f'{v:.3f}' for v in feature_importance['importance'][:8]],
           textposition='auto'),
    row=1, col=2
)

cm = confusion_matrix(y_test, y_pred_best)
fig.add_trace(
    go.Heatmap(z=cm,
               text=cm,
               texttemplate='%{text}',
               colorscale='Blues',
               showscale=False),
    row=1, col=3
)

fig.update_xaxes(title_text="Model", row=1, col=1)
fig.update_yaxes(title_text="F1-Score", row=1, col=1)
fig.update_xaxes(title_text="Importance", row=1, col=2)
fig.update_xaxes(title_text="Predicted", row=1, col=3)
fig.update_yaxes(title_text="Actual", row=1, col=3)

fig.update_layout(
    height=400,
    title_text="<b>Customer Churn Prediction Results</b>",
    showlegend=False
)
fig.show()



Churn Rate: 98.5%
Churned customers: 95,027
Active customers: 1,451

Model Performance Comparison:
--------------------
                     Accuracy  Precision  Recall  F1-Score
Logistic Regression     0.995      0.995   1.000     0.998
Random Forest           1.000      1.000   1.000     1.000
XGBoost                 1.000      1.000   1.000     1.000


In [54]:
clv_data = ml_data.loc[ml_data["Monetary"] < ml_data["Monetary"].quantile(0.95)].copy()
clv_data = clv_data.reset_index(drop=True)

X_clv = clv_data[feature_cols]
y_clv = clv_data["Monetary"]

X_train_clv, X_test_clv, y_train_clv, y_test_clv, idx_train, idx_test = train_test_split(
    X_clv, y_clv, clv_data.index, test_size=0.2, random_state=42
)

rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
rf_regressor.fit(X_train_clv, y_train_clv)

y_pred_clv = rf_regressor.predict(X_test_clv)

mae = mean_absolute_error(y_test_clv, y_pred_clv)
r2 = r2_score(y_test_clv, y_pred_clv)

print("CLV Model Performance:")
print(f"Mean Absolute Error: R${mae:.2f}")
print(f"R\u00b2 Score: {r2:.3f}")
print(f"Average CLV: R${y_clv.mean():.2f}")
print(f"Predicted CLV Range: R${y_pred_clv.min():.2f} - R${y_pred_clv.max():.2f}")

clv_predictions = pd.DataFrame(
    {
        "customer_id": clv_data.loc[idx_test, "customer_id"].values,
        "actual_clv": y_test_clv.values,
        "predicted_clv": y_pred_clv,
        "error": np.abs(y_test_clv.values - y_pred_clv),
    }
)

clv_predictions["clv_segment"] = pd.qcut(
    clv_predictions["predicted_clv"],
    q=4,
    labels=["Low Value", "Medium Value", "High Value", "VIP"],
)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Actual vs Predicted CLV", "CLV Distribution by Segment"),
    horizontal_spacing=0.18,
)

fig.add_trace(
    go.Scatter(
        x=clv_predictions["actual_clv"],
        y=clv_predictions["predicted_clv"],
        mode="markers",
        marker=dict(
            color=clv_predictions["error"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="Error", x=0.47, len=0.85),
            size=7,
        ),
        text=clv_predictions["clv_segment"],
        hovertemplate="Actual: R$%{x:.2f}<br>Predicted: R$%{y:.2f}<br>%{text}<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)

max_axis = float(max(clv_predictions["actual_clv"].max(), clv_predictions["predicted_clv"].max()))
fig.add_trace(
    go.Scatter(
        x=[0, max_axis],
        y=[0, max_axis],
        mode="lines",
        line=dict(color="red", dash="dash"),
        showlegend=False,
        hoverinfo="skip",
    ),
    row=1,
    col=1,
)

for segment in ["Low Value", "Medium Value", "High Value", "VIP"]:
    segment_data = clv_predictions.loc[clv_predictions["clv_segment"] == segment, "predicted_clv"]
    fig.add_trace(go.Box(y=segment_data, name=segment, showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="Actual CLV (R$)", range=[0, max_axis], row=1, col=1)
fig.update_yaxes(title_text="Predicted CLV (R$)", range=[0, max_axis], row=1, col=1)
fig.update_yaxes(title_text="Predicted CLV (R$)", row=1, col=2)

fig.update_layout(
    height=420,
    title_text="<b>Customer Lifetime Value Prediction</b>",
    title_x=0.5,
    showlegend=False,
    margin=dict(l=70, r=140, t=90, b=60),
)
fig.show()

print(clv_predictions["clv_segment"].value_counts())
print(
    f"VIP customers (top 25%) predicted value: R${clv_predictions.loc[clv_predictions['clv_segment']=='VIP','predicted_clv'].mean():.2f}"
)


CLV Model Performance:
Mean Absolute Error: R$0.03
R² Score: 1.000
Average CLV: R$123.83
Predicted CLV Range: R$11.87 - R$449.77


clv_segment
Low Value       4593
VIP             4583
Medium Value    4582
High Value      4573
Name: count, dtype: int64
VIP customers (top 25%) predicted value: R$244.31


In [55]:
daily_sales = completed_orders.groupby(
    completed_orders['order_purchase_timestamp'].dt.date
).agg({
    'order_id': 'count',
    'total_payment': 'sum'
}).rename(columns={'order_id': 'orders', 'total_payment': 'revenue'})

daily_sales.index = pd.to_datetime(daily_sales.index)
daily_sales = daily_sales.sort_index()

window_sizes = [7, 14, 30]
for window in window_sizes:
    daily_sales[f'MA_{window}'] = daily_sales['revenue'].rolling(window=window).mean()

x = np.arange(len(daily_sales))
slope, intercept, r_value, p_value, std_err = stats.linregress(x, daily_sales['revenue'])

daily_sales['trend'] = slope * x + intercept

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Daily Revenue with Moving Averages', 'Order Volume Trend',
                   'Weekly Seasonality', 'Revenue Distribution'),
    specs=[[{'secondary_y': False}, {'secondary_y': False}],
           [{'type': 'bar'}, {'type': 'histogram'}]]
)

fig.add_trace(
    go.Scatter(x=daily_sales.index,
              y=daily_sales['revenue'],
              mode='lines',
              name='Actual',
              line=dict(color='lightgray', width=1)),
    row=1, col=1
)

for window, color in zip([7, 30], ['blue', 'red']):
    fig.add_trace(
        go.Scatter(x=daily_sales.index,
                  y=daily_sales[f'MA_{window}'],
                  mode='lines',
                  name=f'{window}-day MA',
                  line=dict(color=color, width=2)),
        row=1, col=1
    )

fig.add_trace(
    go.Scatter(x=daily_sales.index,
              y=daily_sales['orders'],
              mode='lines',
              fill='tozeroy',
              fillcolor='rgba(0,100,255,0.2)',
              line=dict(color='blue')),
    row=1, col=2
)

daily_sales['weekday'] = pd.to_datetime(daily_sales.index).dayofweek
weekly_avg = daily_sales.groupby('weekday')['revenue'].mean()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig.add_trace(
    go.Bar(x=days,
           y=weekly_avg.values,
           marker_color=['green' if v > weekly_avg.mean() else 'orange' for v in weekly_avg.values]),
    row=2, col=1
)

fig.add_trace(
    go.Histogram(x=daily_sales['revenue'],
                nbinsx=30,
                marker_color='purple'),
    row=2, col=2
)

fig.update_xaxes(title_text="Date", row=1, col=1)
fig.update_xaxes(title_text="Date", row=1, col=2)
fig.update_xaxes(title_text="Day of Week", row=2, col=1)
fig.update_xaxes(title_text="Daily Revenue (R$)", row=2, col=2)
fig.update_yaxes(title_text="Revenue (R$)", row=1, col=1)
fig.update_yaxes(title_text="Orders", row=1, col=2)
fig.update_yaxes(title_text="Avg Revenue (R$)", row=2, col=1)
fig.update_yaxes(title_text="Frequency", row=2, col=2)

fig.update_layout(
    height=700,
    title_text="<b>Demand Forecasting Analysis</b>",
    showlegend=True
)
fig.show()


days_ahead = 30
future_revenue = slope * (len(daily_sales) + days_ahead) + intercept
print(f"30-Day Projection: R${future_revenue:,.2f} daily average")

30-Day Projection: R$43,418.59 daily average


In [56]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Churn Risk Distribution", "CLV Segments", "Feature Importance", "Model Performance"),
    specs=[[{"type": "pie"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "scatter"}]],
    horizontal_spacing=0.16,
    vertical_spacing=0.18
)

churn_dist = ml_data["is_churned"].value_counts()
fig.add_trace(
    go.Pie(
        labels=["Active", "Churned"],
        values=churn_dist.reindex([0, 1], fill_value=0).values if set(churn_dist.index) <= {0, 1} else churn_dist.values,
        hole=0.4,
        marker_colors=["green", "red"],
        textinfo="percent+label",
        textposition="inside",
        sort=False
    ),
    row=1, col=1
)

clv_segment_avg = clv_predictions.groupby("clv_segment")["predicted_clv"].mean().sort_values()
fig.add_trace(
    go.Bar(
        x=clv_segment_avg.values,
        y=clv_segment_avg.index.astype(str),
        orientation="h",
        marker_color=["red", "orange", "lightgreen", "green"],
        text=[f"R${v:.0f}" for v in clv_segment_avg.values],
        textposition="outside",
        cliponaxis=False
    ),
    row=1, col=2
)

fig.add_trace(
    go.Bar(
        y=feature_importance["feature"][:5].astype(str),
        x=feature_importance["importance"][:5],
        orientation="h",
        marker_color="teal",
        cliponaxis=False
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=["Churn F1", "CLV R\u00b2", "Accuracy"],
        y=[results["XGBoost"]["F1-Score"], r2, results["XGBoost"]["Accuracy"]],
        mode="markers+lines",
        marker=dict(size=14),
        line=dict(dash="dash"),
        showlegend=False
    ),
    row=2, col=2
)

fig.update_xaxes(title_text="Average CLV (R$)", row=1, col=2)
fig.update_xaxes(title_text="Importance", row=2, col=1)
fig.update_xaxes(title_text="Metric", row=2, col=2)
fig.update_yaxes(title_text="Score", row=2, col=2)

fig.update_layout(
    height=680,
    title_text="<b>Predictive Analytics Dashboard</b>",
    title_x=0.5,
    showlegend=False,
    margin=dict(l=70, r=140, t=110, b=70)
)

fig.update_annotations(yshift=12)

fig.show()
